# EDA — Dataset SKU-110K

Exploración del dataset antes del entrenamiento: distribuciones de objetos por
imagen, tamaños de bbox y estructura de las anotaciones.

**Rutas de este notebook:** todo cuelga de `ROOT_DIR`, detectado igual que en
`training/pipeline_completo.ipynb`. El notebook reside en
`data_pipeline/notebooks/`, por lo que la raíz del repo se localiza un nivel
arriba.

**Credenciales:** se leen de `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY`
(`.env` en la raíz, o `~/.aws/credentials`). Nunca se deben incluir en el repositorio.


In [ ]:
# --- Setup: raíz del repo, dependencias y credenciales ----------------------
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/vchacin/AnyoneAI-object-detection.git"

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "boto3", "pandas", "pillow", "matplotlib",
                    "opencv-python-headless", "pyyaml", "tqdm"], check=True)
    ROOT_DIR = Path("/content/repo")
    if not ROOT_DIR.exists():
        subprocess.run(["git", "clone", "--branch", "dev", "--single-branch",
                        REPO_URL, str(ROOT_DIR)], check=True)
else:
    ROOT_DIR = Path.cwd()
    if not (ROOT_DIR / "data_pipeline").is_dir() and (ROOT_DIR.parent / "data_pipeline").is_dir():
        ROOT_DIR = ROOT_DIR.parent
    print("Entorno local. Instalar dependencias con: pip install -r requirements.txt")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

DATA_DIR = ROOT_DIR / "data_pipeline" / "dataset_sku110k"
IMAGES_DIR = DATA_DIR / "images"
ANNOT_DIR = DATA_DIR / "annotations"
REPORTS_DIR = ROOT_DIR / "training" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT_DIR =", ROOT_DIR)


In [ ]:
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

if not os.environ.get("AWS_ACCESS_KEY_ID"):
    dotenv = ROOT_DIR / ".env"
    if dotenv.is_file():
        for line in dotenv.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, _, v = line.partition("=")
                os.environ.setdefault(k.strip(), v.strip().strip("\"'"))
    print("AWS_ACCESS_KEY_ID definida:", bool(os.environ.get("AWS_ACCESS_KEY_ID")))

s3 = boto3.client("s3")
bucket = "anyoneai-datasets"
prefix = "SKU-110K/SKU110K_fixed/"

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, MaxKeys=20)
for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"])


In [ ]:
# --- Descarga de las anotaciones -------------------------------------------
ANNOT_DIR.mkdir(parents=True, exist_ok=True)

for split in ("train", "val", "test"):
    key = f"{prefix}annotations/annotations_{split}.csv"
    local_path = ANNOT_DIR / f"annotations_{split}.csv"
    if not local_path.is_file():
        s3.download_file(bucket, key, str(local_path))
    print("Disponible:", local_path.name)


In [ ]:
# --- Carga a pandas -------------------------------------------------------
columns = ["image_name", "x1", "y1", "x2", "y2", "class",
           "image_width", "image_height"]

df_train = pd.read_csv(ANNOT_DIR / "annotations_train.csv", header=None, names=columns)
df_val = pd.read_csv(ANNOT_DIR / "annotations_val.csv", header=None, names=columns)
df_test = pd.read_csv(ANNOT_DIR / "annotations_test.csv", header=None, names=columns)

print("Train:", df_train.shape)
print("Val:", df_val.shape)
print("Test:", df_test.shape)

df_train.head()


In [ ]:
# --- Distribuciones --------------------------------------------------------
n_images_train = df_train["image_name"].nunique()
boxes_per_image = df_train.groupby("image_name").size()

print("Clases unicas:", df_train["class"].unique())
print("Imagenes unicas en train:", n_images_train)
print("Objetos por imagen -> min:", boxes_per_image.min(),
      "max:", boxes_per_image.max(),
      "promedio:", round(boxes_per_image.mean(), 1))

print("\nTamanos de imagen unicos:")
print(df_train[["image_width", "image_height"]].drop_duplicates().shape[0], "combinaciones distintas")
df_train[["image_width", "image_height"]].value_counts().head()


In [ ]:
# --- Visualización de cajas ------------------------------------------------
IMAGES_DIR.mkdir(parents=True, exist_ok=True)


def plot_image_with_boxes(image_name, df, figsize=(12, 12)):
    from PIL import Image

    image = Image.open(IMAGES_DIR / image_name)
    boxes = df[df["image_name"] == image_name]

    fig, ax = plt.subplots(1, figsize=figsize)
    ax.imshow(image)
    for _, row in boxes.iterrows():
        ax.add_patch(patches.Rectangle(
            (row["x1"], row["y1"]), row["x2"] - row["x1"], row["y2"] - row["y1"],
            linewidth=1, edgecolor="red", facecolor="none"))
    ax.set_title(f"{image_name} - {len(boxes)} objetos")
    ax.axis("off")
    plt.show()


In [ ]:
# --- Muestra de imágenes ---------------------------------------------------
sample_images = [boxes_per_image.idxmin(), boxes_per_image.idxmax(), "train_57.jpg"]

for img_name in sample_images:
    key = f"{prefix}images/{img_name}"
    local_path = IMAGES_DIR / img_name
    if not local_path.is_file():
        s3.download_file(bucket, key, str(local_path))
    print("Descargada:", img_name)

plot_image_with_boxes(boxes_per_image.idxmin(), df_train)
plot_image_with_boxes(boxes_per_image.idxmax(), df_train)
plot_image_with_boxes("train_57.jpg", df_train)


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(boxes_per_image, bins=50, color="steelblue", edgecolor="black")
plt.xlabel("Objetos por imagen")
plt.ylabel("Cantidad de imagenes")
plt.title("Distribucion de objetos por imagen (train)")
plt.show()

print(boxes_per_image.describe())


In [ ]:
# Las imágenes con pocos objetos son candidatas a producir falsos positivos de
# hueco, por lo que conviene examinarlas por separado.
low_density = boxes_per_image[boxes_per_image < 10].sort_values()
print("Imagenes con menos de 10 objetos:", len(low_density))
print(low_density)


In [ ]:
df_train["bbox_width"] = df_train["x2"] - df_train["x1"]
df_train["bbox_height"] = df_train["y2"] - df_train["y1"]
df_train["bbox_area"] = df_train["bbox_width"] * df_train["bbox_height"]

print(df_train[["bbox_width", "bbox_height", "bbox_area"]].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_train["bbox_width"], bins=50, color="salmon", edgecolor="black")
axes[0].set_title("Distribucion de ancho de bbox")
axes[1].hist(df_train["bbox_height"], bins=50, color="seagreen", edgecolor="black")
axes[1].set_title("Distribucion de alto de bbox")
plt.show()


In [ ]:
# El pitch (ancho mediano de producto) es el parámetro que utiliza
# api/gap_heuristic.py para determinar el ancho de "un producto faltante".
df_train["bbox_width_rel"] = df_train["bbox_width"] / df_train["image_width"]
df_train["bbox_height_rel"] = df_train["bbox_height"] / df_train["image_height"]
df_train["bbox_area_rel"] = df_train["bbox_width_rel"] * df_train["bbox_height_rel"]

print(df_train[["bbox_width_rel", "bbox_height_rel", "bbox_area_rel"]].describe())
print("\nPitch mediano (px absolutos):", df_train["bbox_width"].median())


In [ ]:
paginator = s3.get_paginator("list_objects_v2")
image_keys = []
for page in paginator.paginate(Bucket=bucket, Prefix=f"{prefix}images/"):
    for obj in page.get("Contents", []):
        image_keys.append(obj["Key"])

print("Total de imagenes en el bucket:", len(image_keys))


In [ ]:
# Muestra concreta para iterar rápidamente: 10 imágenes de train.
sample_names = (df_train["image_name"].drop_duplicates()
                .sample(n=10, random_state=42).tolist())
for image_name in sample_names:
    local_path = IMAGES_DIR / image_name
    if not local_path.is_file():
        s3.download_file(bucket, f"{prefix}images/{image_name}", str(local_path))

print("Filas de anotaciones:", len(df_train))
print("Imagenes descargadas:", len(sample_names))
sample_names


In [ ]:
# Vista previa de la heurística sobre las cajas anotadas (sin modelo): el
# proceso es idéntico a labeling/generate_missing_labels.py.
import subprocess

from api.gap_heuristic import score_candidates  # noqa: F401  (import de humo)

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
out_dir = DATA_DIR / "missing_eda"
cmd = [sys.executable, str(ROOT_DIR / "labeling" / "generate_missing_labels.py"),
       "--images-dir", str(IMAGES_DIR),
       "--csv", str(ANNOT_DIR / "annotations_train.csv"),
       "--glob", "*.jpg", "--limit", "10", "--out", str(out_dir), "--no-coco"]
subprocess.run(cmd, check=True)


In [ ]:
summary = json.loads((DATA_DIR / "missing_eda" / "summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))

manifest = json.loads((DATA_DIR / "missing_eda" / "manifest.json").read_text(encoding="utf-8"))
rows = []
for image in manifest["images"]:
    for c in image["candidates"]:
        rows.append({"imagen": image["name"], "score": c["score"], "supero_umbral": c["auto"]})

candidates_df = pd.DataFrame(rows)
if not candidates_df.empty:
    display(candidates_df.groupby("imagen").agg(
        propuestas=("score", "size"),
        sobre_umbral=("supero_umbral", "sum"),
        score_promedio=("score", "mean")).round(2))


In [ ]:
from IPython.display import display
from PIL import Image

overlay_dir = DATA_DIR / "missing_eda" / "overlays"
paths = sorted(overlay_dir.glob("*.jpg"))
print("Overlays:", len(paths))
for p in paths[:3]:
    display(Image.open(p))


## Siguiente paso

Con estas distribuciones se puede calibrar `api/gap_heuristic.py`: el
`row_pitch` se basa en el ancho mediano de bbox, y los umbrales
(`GAP_MIN_FRAC`, `AUTO_SCORE`) se ajustan examinando los overlays de la
celda anterior.

Para pasar a etiquetado y entrenamiento, seguir con
`training/pipeline_completo.ipynb`.
